In [ ]:
# 0) Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Optional for hypothesis testing (fallback to NumPy-only if not available)
try:
    from scipy import stats
    SCIPY_OK = True
except Exception:
    SCIPY_OK = False

pd.set_option("display.max_columns", 50)

# 1) Data Import and Cleaning
url = "https://raw.githubusercontent.com/user257814938/PSTB-DI-Bootcamp/refs/heads/main/Week3/Day2/DailyChallenge/global_power_plant_database.csv"
df = pd.read_csv(url)

# Basic overview
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nMissing values (top 20):")
print(df.isna().sum().sort_values(ascending=False).head(20))

# Relevant columns (adapt to your dataset columns)
cols_keep = [c for c in [
    "name", "country", "country_long", "capacity_mw",
    "latitude", "longitude", "primary_fuel", "commissioning_year"
] if c in df.columns]
df = df[cols_keep].copy()

# Convert types (NumPy used via pandas)
for col in ["capacity_mw", "latitude", "longitude", "commissioning_year"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Handle missing values
# Strategy:
# - latitude/longitude/capacity: drop rows missing core geography/capacity (for mapping/stats)
# - primary_fuel/country: drop rows if missing (for grouping)
# - commissioning_year: keep NaN for non time-series parts; fill later for trend plots if needed
base_len = len(df)
df = df.dropna(subset=["primary_fuel", "country_long"])
print("\nRows dropped due to missing primary_fuel/country_long:", base_len - len(df))

# Duplicates
dups = df.duplicated().sum()
df = df.drop_duplicates()
print("Duplicates removed:", dups)

# 2) Exploratory Data Analysis
num_cols = [c for c in ["capacity_mw", "latitude", "longitude"] if c in df.columns]
print("\nNumeric summary:")
print(df[num_cols].describe())

# Distribution by country and fuel type
print("\nTop countries by plant count:")
print(df["country_long"].value_counts().head(10))
print("\nTop fuels by plant count:")
print(df["primary_fuel"].value_counts().head(10))

# 3) Statistical Analysis
# Power output by fuel type
if "capacity_mw" in df.columns and "primary_fuel" in df.columns:
    fuel_groups = df.dropna(subset=["capacity_mw"]).groupby("primary_fuel")["capacity_mw"]
    cap_stats = fuel_groups.agg(["count", "mean", "median", "std"]).sort_values("mean", ascending=False)
    print("\nCapacity by fuel type (count, mean, median, std):")
    print(cap_stats.head(15))

    # Hypothesis test: mean capacity differs between top fuels
    top_fuels = cap_stats.head(5).index.tolist()
    samples = [df.loc[df["primary_fuel"] == f, "capacity_mw"].dropna().values for f in top_fuels if df.loc[df["primary_fuel"] == f, "capacity_mw"].notna().sum() > 2]

    if len(samples) >= 2:
        if SCIPY_OK:
            f_stat, p_val = stats.f_oneway(*samples)
            print(f"\nANOVA across top fuels (k={len(samples)}): F={f_stat:.3f}, p={p_val:.3e}")
        else:
            # NumPy-only fallback: pairwise Welch-esque t-tests (illustrative)
            def np_ttest(a, b):
                m1, m2 = a.mean(), b.mean()
                v1, v2 = a.var(ddof=1), b.var(ddof=1)
                n1, n2 = len(a), len(b)
                t = (m1 - m2) / np.sqrt(v1/n1 + v2/n2)
                return t
            print("\nSciPy not available; showing pairwise t-statistics (no p-values):")
            for i in range(len(samples)):
                for j in range(i+1, len(samples)):
                    t = np_ttest(samples[i], samples[j])
                    print(f"t({top_fuels[i]} vs {top_fuels[j]}): {t:.3f}")

# 4) Time Series Analysis
if "commissioning_year" in df.columns:
    ts = df.dropna(subset=["commissioning_year"]).copy()
    ts["commissioning_year"] = ts["commissioning_year"].astype(int)
    yearly_count = ts.groupby("commissioning_year")["name"].count()
    print("\nCommissioned plants per year (sample):")
    print(yearly_count.head())

    if "primary_fuel" in ts.columns:
        mix = ts.groupby(["commissioning_year", "primary_fuel"])["name"].count().reset_index()
        mix_pivot = mix.pivot(index="commissioning_year", columns="primary_fuel", values="name").fillna(0)

# 5) Advanced Visualization
sns.set(style="whitegrid")

# Monthly/Year trend: plants commissioned per year
if "commissioning_year" in df.columns and len(yearly_count) > 0:
    plt.figure(figsize=(10, 4))
    plt.plot(yearly_count.index, yearly_count.values, marker="o")
    plt.title("Plants commissioned per year")
    plt.xlabel("Year")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

# Fuel mix over years (stacked area)
if "commissioning_year" in df.columns and "primary_fuel" in df.columns and 'mix_pivot' in locals():
    plt.figure(figsize=(12, 5))
    plt.stackplot(mix_pivot.index, mix_pivot.T.values, labels=mix_pivot.columns)
    plt.title("Fuel mix by commissioning year (count of plants)")
    plt.xlabel("Year")
    plt.ylabel("Count")
    plt.legend(loc="upper left", ncol=2, fontsize=8)
    plt.tight_layout()
    plt.show()

# Geographic distribution (lon/lat)
if {"latitude", "longitude"}.issubset(df.columns):
    subset = df.dropna(subset=["latitude", "longitude"]).copy()
    c = None
    if "primary_fuel" in subset.columns:
        # Encode fuel categories into integers for coloring with Matplotlib
        subset["fuel_code"] = subset["primary_fuel"].astype("category").cat.codes
        c = subset["fuel_code"]
    plt.figure(figsize=(8, 6))
    plt.scatter(subset["longitude"], subset["latitude"], c=c, s=np.clip(subset["capacity_mw"].fillna(0), 5, 200)*0.2, alpha=0.6)
    plt.title("Geographical distribution of power plants")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.tight_layout()
    plt.show()

# 6) Matrix Operations in Real-World Context
# Example: relationship among capacity, latitude, longitude via covariance/eigen decomposition
matrix_cols = [c for c in ["capacity_mw", "latitude", "longitude"] if c in df.columns]
X = df.dropna(subset=matrix_cols)[matrix_cols].values
if X.shape[0] > 5:
    # Standardize with NumPy
    X_std = (X - X.mean(axis=0)) / X.std(axis=0, ddof=1)
    # Covariance matrix and eigen decomposition
    cov = np.cov(X_std, rowvar=False)
    eigvals, eigvecs = np.linalg.eig(cov)
    order = np.argsort(eigvals)[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]
    print("\nCovariance matrix (standardized):")
    print(np.round(cov, 3))
    print("\nEigenvalues (variance explained):")
    print(np.round(eigvals, 3))
    print("\nFirst eigenvector (principal direction):")
    print(np.round(eigvecs[:, 0], 3))

# 7) Integrating NumPy with Pandas and Matplotlib
# Example: NumPy-based complex filtering used inside Pandas workflow
if "capacity_mw" in df.columns:
    cap_thr = np.nanpercentile(df["capacity_mw"], 90)  # top 10% by capacity
    mask = df["capacity_mw"].to_numpy() >= cap_thr
    top_plants = df.loc[mask, ["name", "country_long", "primary_fuel", "capacity_mw"]].sort_values("capacity_mw", ascending=False).head(10)
    print("\nTop plants by capacity (NumPy mask):")
    print(top_plants)

    # Simple visualization of top capacities
    plt.figure(figsize=(10, 4))
    plt.bar(top_plants["name"], top_plants["capacity_mw"])
    plt.title("Top capacities")
    plt.xlabel("Plant")
    plt.ylabel("Capacity (MW)")
    plt.xticks(rotation=60, ha="right")
    plt.tight_layout()
    plt.show()
